<a href="https://colab.research.google.com/github/DorcasOpatola/ai-ml_foundations/blob/main/tinyshakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64   # how many independent sequences to be processed in parallel, 1024
block_size = 256     # maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# --------------

In [8]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-06 12:20:06--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-11-06 12:20:06 (33.8 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [12]:

# read the data from file
torch.manual_seed(1337)
with open('/content/input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

# Encode and Decode
# all the unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i, ch in enumerate(chars) }
itos = { i:ch for i, ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]     # encoder: takes a list of strings, outputs a list of integers
decode = lambda l: ''.join([itos[i] for i in l])     # decoder: takes a list of integers, outputs a string

text = text[:200000]
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))  # first 90% will be train, the rest for evaluation
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
  # generate a small batch of data of inputs x and targets y
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

@torch.no_grad()     # context manager, to inform PyTorch to not perform back propagation
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


# head module
class Head(nn.Module):
  """one head of self-attention"""

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)     # (B,T,C)
    q = self.query(x)   # (B,T,C)
    # compute attention scores ("affinities")
    wei = q @ k.transpose(-2, -1) * C**-0.5   # (B, T, C) @ (B, C, T) ---> (B, T, T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))    #(B, T, T)
    wei = F.softmax(wei, dim=-1)    #(B, T, T)
    wei = self.dropout(wei)

    # perform the weighted aggregation of the values
    v = self.value(x)    #(B, T, C)
    out = wei @ v    # (B, T, T) @ (B, T, C) ---> (B, T, C)
    return out


class MultiHeadAttention(nn.Module):
  """ multiple heads of attention in parallel"""

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """a simple linear layer followed by non-linearity"""

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, 4 * n_embd),
      nn.ReLU(),
      nn.Linear(4 * n_embd, n_embd),
      nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  """ Transformer block: communication followed by computation"""

  def __init__(self, n_embd, n_head):
    # n_embd: embedding dimension,   n_head: the number of heads we want
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


# simple bigram model
class BigramLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)     # n_embd - number of embedding dimensions
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

    self.ln_f = nn.LayerNorm(n_embd)  # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape

    # idx and targets are both (B, T) tensor of integers
    tok_emb = self.token_embedding_table(idx)   # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))     # (T, C)
    x = tok_emb + pos_emb   # (B, T, C)
    x = self.blocks(x)      # (B,T,C)
    x = self.ln_f(x)        # (B,T,C)
    logits = self.lm_head(x)  # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B * T)
      loss = F.cross_entropy(logits, targets)
    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      # get the predictions
      logits, loss = self(idx_cond)
      # focus only on the last time step
      logits = logits[:, -1, :]     # becomes (B, C)
      # apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1)     # (B, C)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1)
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1)     # (B, T+1)
    return idx


model = BigramLanguageModel()
m = model.to(device)

# create a Pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
  # every once in a while, evaluate the loss on train and val datasets
  if iter % eval_interval == 0:
    losses = estimate_loss()
    print(f"step {iter}: train loss {losses['train']: .4f}, val loss {losses['val']: .4f}")

  # sample a batch of data
  xb, yb = get_batch('train')

  # evaluate the loss
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


step 0: train loss  4.2894, val loss  4.2861
step 500: train loss  1.8738, val loss  2.0398
step 1000: train loss  1.4370, val loss  1.7265
step 1500: train loss  1.1923, val loss  1.6603
step 2000: train loss  0.9372, val loss  1.7321
step 2500: train loss  0.6486, val loss  1.9125
step 3000: train loss  0.3878, val loss  2.2500
step 3500: train loss  0.2135, val loss  2.5617
step 4000: train loss  0.1291, val loss  2.9064
step 4500: train loss  0.0977, val loss  3.1163

Better
With Glace thee, bastards and his king;
And my present to his noble queence,
Stain all hour treading in the eye that his
mirth behalf-private, while, and the king
That's in the emberity: and the otherwain man is requests
To know the best of so?

MENENIUS:
Come, my good friends, my chost chamberlain?

BRUTUS:
Well, well, seize him, and we'll be revenged
With our wits every at like a lip. Nan't pleace,
You'll have make by my an en!
First Starving way thou hast to pluck'd at:
Thou matter fro
